In [1]:
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

app = FastAPI(title="Sentiment Analysis API")

# Load your saved model and tokenizer from Task 2
MODEL_PATH = "./saved_model"
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_PATH)
model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

class TextRequest(BaseModel):
    text: str

@app.get("/")
def home():
    return {"message": "Sentiment Analysis API is running!"}

@app.post("/predict")
def predict_sentiment(request: TextRequest):
    inputs = tokenizer(request.text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=1).item()
    
    sentiment = "Positive" if prediction == 1 else "Negative"
    return {"text": request.text, "sentiment": sentiment}

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [3]:
%%writefile requirements.txt
fastapi==0.138.0
uvicorn==0.49.0
pydantic==2.10.3
torch==2.12.1 --index-url https://download.pytorch.org/whl/cpu
transformers==5.12.1

Overwriting requirements.txt


In [4]:
%%writefile Dockerfile
FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
COPY saved_model/ ./saved_model/
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting Dockerfile


In [6]:
import threading
import time
import requests
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
import torch

# 1. Initialize FastAPI App
app = FastAPI()

MODEL_PATH = "./saved_model"
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_PATH)
model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

class TextRequest(BaseModel):
    text: str

@app.post("/predict")
def predict_sentiment(request: TextRequest):
    inputs = tokenizer(request.text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=1).item()
    return {"text": request.text, "sentiment": "Positive" if prediction == 1 else "Negative"}

# 2. Run API in the background using a completely fresh port (8222)
def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8222, log_level="error")

thread = threading.Thread(target=run_api, daemon=True)
thread.start()
time.sleep(3) # Give the server an extra moment to safely boot up

# 3. Send a sample request to get your clean Task 3 Demo output!
url = "http://127.0.0.1:8222/predict"
payload = {"text": "This deployment process was incredibly fast and easy!"}
response = requests.post(url, json=payload)

print("=== Task 3 Demo Output ===")
print("Status Code:", response.status_code)
print("Response JSON:", response.json())

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

=== Task 3 Demo Output ===
Status Code: 200
Response JSON: {'text': 'This deployment process was incredibly fast and easy!', 'sentiment': 'Positive'}
